<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/03_baseline_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Baseline - Random Forest (Versión NumPy 2.0)
**Proyecto:** Detección de Deslizamientos | Universidad EAFIT

Establecemos el rendimiento base usando descriptores HOG sobre imágenes Sentinel-2 y datos de elevación.

## 1. Conexión a Drive y Localización de Datos

In [ ]:
from google.colab import drive
import os, h5py
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
detectar_img = list(base_path.glob('**/TrainData/img'))

if detectar_img:
    train_img_dir = detectar_img[0]
    train_mask_dir = detectar_img[0].parent / "mask"
    img_list = sorted(list(train_img_dir.glob("*.h5")))
    mask_list = sorted(list(train_mask_dir.glob("*.h5")))
    print(f"✅ Datos listos: {len(img_list)} muestras detectadas.")
else:
    print("❌ ERROR: No se encontró la carpeta en Drive.")

## 2. Extracción de Características (HOG + Geometría)
Corregimos la normalización para compatibilidad con NumPy 2.0.

In [ ]:
N_SAMPLES = 1000 
X, y = [], []

for i in tqdm(range(min(N_SAMPLES, len(img_list))), desc="Extrayendo Características"):
    with h5py.File(img_list[i], "r") as hf:
        patch = hf[list(hf.keys())[0]][()]
    with h5py.File(mask_list[i], "r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    
    # --- CORRECCIÓN NUMPY 2.0 ---
    rgb = patch[:,:,[3,2,1]]
    r_min = rgb.min()
    # Usamos np.ptp() como función, no como método del array
    r_range = np.ptp(rgb) 
    
    rgb_norm = ((rgb - r_min) / (r_range + 1e-8) * 255).astype("uint8")
    
    # 1. Textura con HOG
    feats_hog = hog(rgb_norm, pixels_per_cell=(16,16), cells_per_block=(2,2), 
                channel_axis=-1, feature_vector=True)
    
    # 2. Dato Geotécnico: Promedio de Pendiente (Banda 13 -> Índice 12)
    avg_slope = np.mean(patch[:,:,12])
    
    # Unificamos vectores
    combined = np.hstack([feats_hog, avg_slope])
    
    X.append(combined)
    y.append(int(mask.max() > 0))

X, y = np.array(X), np.array(y)
print(f"✓ Dataset listo: {X.shape}")

## 3. Validación Cruzada

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_vals = []

for fold, (t_idx, v_idx) in enumerate(skf.split(X, y)):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X[t_idx], y[t_idx])
    
    p = rf.predict(X[v_idx])
    score = f1_score(y[v_idx], p)
    f1_vals.append(score)
    print(f"Fold {fold+1}: F1 = {score:.4f}")

print(f"\n🚀 RESULTADO BASELINE: {np.mean(f1_vals):.4f} ± {np.std(f1_vals):.4f}")